# Project 3: Machine Learning for Predicting Trading Signals

**Student:** Mohammad Tamim Hamkar


I used the prepared stock market dataset from Project 2 as the starting point for this project. Before calculating the technical indicators, I loaded the dataset and checked its size, columns, data types, missing values, and date range.

## Step 1 — Import Libraries and Setup

I imported the Python libraries needed for data preparation, visualization, machine learning, and model evaluation. Keeping the required libraries together at the beginning makes the notebook organized and easier to reproduce.

In [27]:
# Import Libraries and Setup

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Data preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Cross-validation and model optimization
from sklearn.model_selection import (
    TimeSeriesSplit,
    cross_val_score,
    GridSearchCV
)

# Display settings
pd.set_option("display.max_columns", None)

print("All libraries imported successfully.")

All libraries imported successfully.


## Step 2 — Load and Verify the Prepared Dataset

I loaded the prepared stock market dataset from Project 2 and performed a quick check before starting the technical indicator calculations. I checked the dataset size, columns, missing values, and date range to make sure the data was ready for Project 3.

In [4]:
# Load and Verify the Prepared Dataset

stock_data = pd.read_csv("stock_prepared.csv")

# Convert date column to datetime
stock_data["date"] = pd.to_datetime(stock_data["date"])

print("Dataset Shape:", stock_data.shape)

print("\nColumns:")
print(stock_data.columns.tolist())

print("\nTotal Missing Values:")
print(stock_data.isnull().sum().sum())

print("\nDate Range:")
print("Minimum Date:", stock_data["date"].min())
print("Maximum Date:", stock_data["date"].max())

display(stock_data.head())

Dataset Shape: (892459, 32)

Columns:
['ticker', 'open', 'close', 'adj_close', 'low', 'high', 'volume', 'date', 'price_change', 'daily_return', 'close_lag_1', 'ma_5', 'ma_20', 'volatility_20', 'daily_range_pct', 'name', 'industry', 'exchange_NASDAQ', 'exchange_NYSE', 'sector_BASIC INDUSTRIES', 'sector_CAPITAL GOODS', 'sector_CONSUMER DURABLES', 'sector_CONSUMER NON-DURABLES', 'sector_CONSUMER SERVICES', 'sector_ENERGY', 'sector_FINANCE', 'sector_HEALTH CARE', 'sector_MISCELLANEOUS', 'sector_PUBLIC UTILITIES', 'sector_TECHNOLOGY', 'sector_TRANSPORTATION', 'sector_Unknown']

Total Missing Values:
0

Date Range:
Minimum Date: 1970-11-18 00:00:00
Maximum Date: 2018-08-24 00:00:00


,ticker,open,close,adj_close,low,high,volume,date,price_change,daily_return,close_lag_1,ma_5,ma_20,volatility_20,daily_range_pct,name,industry,exchange_NASDAQ,exchange_NYSE,sector_BASIC INDUSTRIES,sector_CAPITAL GOODS,sector_CONSUMER DURABLES,sector_CONSUMER NON-DURABLES,sector_CONSUMER SERVICES,sector_ENERGY,sector_FINANCE,sector_HEALTH CARE,sector_MISCELLANEOUS,sector_PUBLIC UTILITIES,sector_TECHNOLOGY,sector_TRANSPORTATION,sector_Unknown
0,A,-0.020737,-0.305493,13.385047,-0.020633,-0.020494,0.463485,2001-10-08,-0.680162,-0.147198,-0.219351,-0.165323,0.269444,0.098912,0.090166,"AGILENT TECHNOLOGIES, INC.",BIOTECHNOLOGY: LABORATORY ANALYTICAL INSTRUMENTS,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
1,A,-0.019461,-0.240979,15.497158,-0.019265,-0.019438,0.145409,2001-11-09,0.490285,0.109640,-0.303626,-0.209580,0.228795,0.074925,-0.036653,"AGILENT TECHNOLOGIES, INC.",BIOTECHNOLOGY: LABORATORY ANALYTICAL INSTRUMENTS,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
2,A,-0.018695,-0.199942,16.840664,-0.018544,-0.018611,2.105374,2001-11-28,0.305478,0.054053,-0.239035,-0.219971,0.165396,0.059922,0.000008,"AGILENT TECHNOLOGIES, INC.",BIOTECHNOLOGY: LABORATORY ANALYTICAL INSTRUMENTS,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
3,A,-0.017437,-0.146307,18.596594,-0.017416,-0.017630,0.900875,2001-12-06,0.404642,0.067791,-0.197948,-0.220317,0.095658,0.060006,-0.045157,"AGILENT TECHNOLOGIES, INC.",BIOTECHNOLOGY: LABORATORY ANALYTICAL INSTRUMENTS,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0
4,A,-0.018130,-0.164058,18.015448,-0.017893,-0.017988,0.872020,2001-12-14,-0.157292,-0.038153,-0.144249,-0.208772,0.060133,0.048976,-0.010093,"AGILENT TECHNOLOGIES, INC.",BIOTECHNOLOGY: LABORATORY ANALYTICAL INSTRUMENTS,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0


## Step 3 — Prepare Data for Technical Indicators

Before calculating MACD and RSI, I sorted the dataset by ticker and date. This ensures that each stock's price history is in chronological order so the technical indicators can be calculated correctly for each stock.

In [5]:
# Prepare Data for Technical Indicators

# Sort each stock by ticker and date
stock_data = stock_data.sort_values(
    by=["ticker", "date"]
).reset_index(drop=True)

# Check the result
print("Dataset Shape:", stock_data.shape)
print("Number of Unique Stocks:", stock_data["ticker"].nunique())

print("\nFirst 10 Ticker and Date Records:")
display(stock_data[["ticker", "date", "close"]].head(10))

Dataset Shape: (892459, 32)
Number of Unique Stocks: 5059

First 10 Ticker and Date Records:


,ticker,date,close
0,A,2001-10-08,-0.305493
1,A,2001-11-09,-0.240979
2,A,2001-11-28,-0.199942
3,A,2001-12-06,-0.146307
4,A,2001-12-14,-0.164058
5,A,2002-01-08,-0.089236
6,A,2002-03-22,-0.058315
7,A,2002-06-03,-0.227045
8,A,2002-06-05,-0.235253
9,A,2002-07-09,-0.282016


## Step 4 — Calculate MACD

I manually calculated the MACD indicator for each stock using Pandas. First, I calculated the 12-period and 26-period exponential moving averages (EMA) of the closing price. I then calculated MACD by subtracting EMA26 from EMA12 and calculated the 9-period EMA of MACD as the signal line.

In [6]:
# Calculate MACD Manually

# Calculate 12-period EMA for each stock
stock_data["ema_12"] = stock_data.groupby("ticker")["close"].transform(
    lambda x: x.ewm(span=12, adjust=False).mean()
)

# Calculate 26-period EMA for each stock
stock_data["ema_26"] = stock_data.groupby("ticker")["close"].transform(
    lambda x: x.ewm(span=26, adjust=False).mean()
)

# Calculate MACD
stock_data["macd"] = stock_data["ema_12"] - stock_data["ema_26"]

# Calculate 9-period EMA of MACD for each stock
stock_data["macd_signal_line"] = stock_data.groupby("ticker")["macd"].transform(
    lambda x: x.ewm(span=9, adjust=False).mean()
)

print("MACD calculations completed successfully.")

display(
    stock_data[
        ["ticker", "date", "close", "ema_12",
         "ema_26", "macd", "macd_signal_line"]
    ].head(10)
)

MACD calculations completed successfully.


,ticker,date,close,ema_12,ema_26,macd,macd_signal_line
0,A,2001-10-08,-0.305493,-0.305493,-0.305493,0.000000,0.000000
1,A,2001-11-09,-0.240979,-0.295568,-0.300714,0.005146,0.001029
2,A,2001-11-28,-0.199942,-0.280856,-0.293250,0.012394,0.003302
3,A,2001-12-06,-0.146307,-0.260156,-0.282365,0.022209,0.007083
4,A,2001-12-14,-0.164058,-0.245372,-0.273602,0.028230,0.011313
5,A,2002-01-08,-0.089236,-0.221351,-0.259945,0.038594,0.016769
6,A,2002-03-22,-0.058315,-0.196269,-0.245009,0.048741,0.023163
7,A,2002-06-03,-0.227045,-0.201003,-0.243679,0.042675,0.027066
8,A,2002-06-05,-0.235253,-0.206273,-0.243055,0.036782,0.029009
9,A,2002-07-09,-0.282016,-0.217925,-0.245941,0.028015,0.028810


## Step 5 — Create MACD Trading Signals

I created MACD trading signals by identifying actual crossovers between the MACD line and the signal line. A Buy signal occurs when MACD crosses above the signal line, while a Sell signal occurs when MACD crosses below it. All other observations are classified as Hold.

In [7]:
# Create MACD Buy, Sell, and Hold Signals

# Previous MACD and signal-line values for each stock
prev_macd = stock_data.groupby("ticker")["macd"].shift(1)
prev_signal = stock_data.groupby("ticker")["macd_signal_line"].shift(1)

# Identify actual MACD crossovers
buy_condition = (
    (stock_data["macd"] > stock_data["macd_signal_line"]) &
    (prev_macd <= prev_signal)
)

sell_condition = (
    (stock_data["macd"] < stock_data["macd_signal_line"]) &
    (prev_macd >= prev_signal)
)

# Create MACD signal
stock_data["macd_signal"] = np.select(
    [buy_condition, sell_condition],
    ["Buy", "Sell"],
    default="Hold"
)

print("MACD signals created successfully.")

print("\nMACD Signal Counts:")
print(stock_data["macd_signal"].value_counts())

display(
    stock_data[
        ["ticker", "date", "macd",
         "macd_signal_line", "macd_signal"]
    ].head(15)
)

MACD signals created successfully.

MACD Signal Counts:
macd_signal
Hold    821368
Buy      35708
Sell     35383
Name: count, dtype: int64


,ticker,date,macd,macd_signal_line,macd_signal
0,A,2001-10-08,0.000000,0.000000,Hold
1,A,2001-11-09,0.005146,0.001029,Buy
2,A,2001-11-28,0.012394,0.003302,Hold
3,A,2001-12-06,0.022209,0.007083,Hold
4,A,2001-12-14,0.028230,0.011313,Hold
5,A,2002-01-08,0.038594,0.016769,Hold
6,A,2002-03-22,0.048741,0.023163,Hold
7,A,2002-06-03,0.042675,0.027066,Hold
8,A,2002-06-05,0.036782,0.029009,Hold
9,A,2002-07-09,0.028015,0.028810,Sell


## Step 6 — Calculate RSI

I manually calculated the Relative Strength Index (RSI) for each stock. I first calculated the change in closing price, separated positive changes into gains and negative changes into losses, and then used exponential moving averages to calculate the relative strength (RS) and RSI.

In [8]:
# Calculate RSI Manually

# Calculate price change separately for each stock
stock_data["rsi_change"] = stock_data.groupby("ticker")["close"].diff()

# Separate gains and losses
stock_data["gain"] = stock_data["rsi_change"].clip(lower=0)
stock_data["loss"] = -stock_data["rsi_change"].clip(upper=0)

# Calculate 14-period exponential average gain and loss
stock_data["avg_gain"] = stock_data.groupby("ticker")["gain"].transform(
    lambda x: x.ewm(span=14, adjust=False).mean()
)

stock_data["avg_loss"] = stock_data.groupby("ticker")["loss"].transform(
    lambda x: x.ewm(span=14, adjust=False).mean()
)

# Calculate Relative Strength
stock_data["rs"] = stock_data["avg_gain"] / stock_data["avg_loss"]

# Calculate RSI
stock_data["rsi"] = 100 - (100 / (1 + stock_data["rs"]))

print("RSI calculations completed successfully.")

display(
    stock_data[
        ["ticker", "date", "close", "rsi_change",
         "gain", "loss", "avg_gain", "avg_loss", "rsi"]
    ].head(15)
)

RSI calculations completed successfully.


,ticker,date,close,rsi_change,gain,loss,avg_gain,avg_loss,rsi
0,A,2001-10-08,-0.305493,NaN,NaN,NaN,NaN,NaN,NaN
1,A,2001-11-09,-0.240979,0.064514,0.064514,-0.000000,0.064514,-0.000000,100.000000
2,A,2001-11-28,-0.199942,0.041037,0.041037,-0.000000,0.061384,-0.000000,100.000000
3,A,2001-12-06,-0.146307,0.053635,0.053635,-0.000000,0.060351,-0.000000,100.000000
4,A,2001-12-14,-0.164058,-0.017751,0.000000,0.017751,0.052304,0.002367,95.670825
5,A,2002-01-08,-0.089236,0.074821,0.074821,-0.000000,0.055306,0.002051,96.423796
6,A,2002-03-22,-0.058315,0.030921,0.030921,-0.000000,0.052055,0.001778,96.697682
7,A,2002-06-03,-0.227045,-0.168730,0.000000,0.168730,0.045114,0.024038,65.239049
8,A,2002-06-05,-0.235253,-0.008207,0.000000,0.008207,0.039099,0.021927,64.069178
9,A,2002-07-09,-0.282016,-0.046763,0.000000,0.046763,0.033886,0.025239,57.312627


## Step 7 — Create RSI Trading Signals

I created RSI trading signals using the RSI levels calculated in the previous step. An RSI below 30 is considered a Buy signal, while an RSI above 70 is considered a Sell signal. Values between 30 and 70 are classified as Hold.

In [9]:
# Create RSI Buy, Sell, and Hold Signals

stock_data["rsi_signal"] = "Hold"

stock_data.loc[stock_data["rsi"] < 30, "rsi_signal"] = "Buy"
stock_data.loc[stock_data["rsi"] > 70, "rsi_signal"] = "Sell"

print("RSI signals created successfully.")

print("\nRSI Signal Counts:")
print(stock_data["rsi_signal"].value_counts())

display(
    stock_data[
        ["ticker", "date", "rsi", "rsi_signal"]
    ].head(15)
)

RSI signals created successfully.

RSI Signal Counts:
rsi_signal
Hold    601423
Sell    193645
Buy      97391
Name: count, dtype: int64


,ticker,date,rsi,rsi_signal
0,A,2001-10-08,NaN,Hold
1,A,2001-11-09,100.000000,Sell
2,A,2001-11-28,100.000000,Sell
3,A,2001-12-06,100.000000,Sell
4,A,2001-12-14,95.670825,Sell
5,A,2002-01-08,96.423796,Sell
6,A,2002-03-22,96.697682,Sell
7,A,2002-06-03,65.239049,Hold
8,A,2002-06-05,64.069178,Hold
9,A,2002-07-09,57.312627,Hold


## Step 8 — Create Final Trading Signal

I combined the MACD and RSI signals to create the final trading signal. A Buy signal is assigned only when both MACD and RSI indicate Buy, and a Sell signal is assigned only when both indicate Sell. All other cases are classified as Hold.

In [10]:
#  Create Final Trading Signal

buy_condition = (
    (stock_data["macd_signal"] == "Buy") &
    (stock_data["rsi_signal"] == "Buy")
)

sell_condition = (
    (stock_data["macd_signal"] == "Sell") &
    (stock_data["rsi_signal"] == "Sell")
)

stock_data["signal"] = np.select(
    [buy_condition, sell_condition],
    ["Buy", "Sell"],
    default="Hold"
)

print("Final trading signals created successfully.")

print("\nFinal Signal Counts:")
print(stock_data["signal"].value_counts())

display(
    stock_data[
        ["ticker", "date", "macd_signal",
         "rsi", "rsi_signal", "signal"]
    ].head(20)
)

Final trading signals created successfully.

Final Signal Counts:
signal
Hold    890575
Buy       1131
Sell       753
Name: count, dtype: int64


,ticker,date,macd_signal,rsi,rsi_signal,signal
0,A,2001-10-08,Hold,NaN,Hold,Hold
1,A,2001-11-09,Buy,100.000000,Sell,Hold
2,A,2001-11-28,Hold,100.000000,Sell,Hold
3,A,2001-12-06,Hold,100.000000,Sell,Hold
4,A,2001-12-14,Hold,95.670825,Sell,Hold
5,A,2002-01-08,Hold,96.423796,Sell,Hold
6,A,2002-03-22,Hold,96.697682,Sell,Hold
7,A,2002-06-03,Hold,65.239049,Hold,Hold
8,A,2002-06-05,Hold,64.069178,Hold,Hold
9,A,2002-07-09,Sell,57.312627,Hold,Hold


## Step 9 — Check Final Signal Distribution

I checked the distribution of the final trading signals before preparing the data for machine learning. This helps show how often Buy, Sell, and Hold signals occur in the dataset and identifies any class imbalance.

In [11]:
# Check Final Signal Distribution

signal_counts = stock_data["signal"].value_counts()
signal_percentages = (
    stock_data["signal"].value_counts(normalize=True) * 100
).round(2)

signal_summary = pd.DataFrame({
    "Count": signal_counts,
    "Percentage": signal_percentages
})

print("Final Trading Signal Distribution:")
display(signal_summary)

Final Trading Signal Distribution:


,Count,Percentage
signal,,
Hold,890575,99.79
Buy,1131,0.13
Sell,753,0.08


## Step 10 — Prepare Features and Target

I prepared the dataset for machine learning by selecting numerical market and technical-indicator features as the input variables. The final trading signal was used as the target variable. Signal columns used to create the target were excluded from the input features to avoid data leakage.

In [12]:
# Prepare Features and Target

feature_columns = [
    "open",
    "close",
    "low",
    "high",
    "volume",
    "price_change",
    "daily_return",
    "close_lag_1",
    "ma_5",
    "ma_20",
    "volatility_20",
    "daily_range_pct",
    "ema_12",
    "ema_26",
    "macd",
    "macd_signal_line",
    "rsi"
]

# Create feature matrix and target
X = stock_data[feature_columns].copy()
y = stock_data["signal"].copy()

# Remove rows with missing values created by RSI
valid_rows = X.notna().all(axis=1)

X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

print("Features and target prepared successfully.")
print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nTarget Distribution:")
print(y.value_counts())

print("\nMissing Values in X:", X.isnull().sum().sum())

display(X.head())

Features and target prepared successfully.
X Shape: (881371, 17)
y Shape: (881371,)

Target Distribution:
signal
Hold    879487
Buy       1131
Sell       753
Name: count, dtype: int64

Missing Values in X: 0


,open,close,low,high,volume,price_change,daily_return,close_lag_1,ma_5,ma_20,volatility_20,daily_range_pct,ema_12,ema_26,macd,macd_signal_line,rsi
0,-0.019461,-0.240979,-0.019265,-0.019438,0.145409,0.490285,0.109640,-0.303626,-0.209580,0.228795,0.074925,-0.036653,-0.295568,-0.300714,0.005146,0.001029,100.000000
1,-0.018695,-0.199942,-0.018544,-0.018611,2.105374,0.305478,0.054053,-0.239035,-0.219971,0.165396,0.059922,0.000008,-0.280856,-0.293250,0.012394,0.003302,100.000000
2,-0.017437,-0.146307,-0.017416,-0.017630,0.900875,0.404642,0.067791,-0.197948,-0.220317,0.095658,0.060006,-0.045157,-0.260156,-0.282365,0.022209,0.007083,100.000000
3,-0.018130,-0.164058,-0.017893,-0.017988,0.872020,-0.157292,-0.038153,-0.144249,-0.208772,0.060133,0.048976,-0.010093,-0.245372,-0.273602,0.028230,0.011313,95.670825
4,-0.016452,-0.089236,-0.016325,-0.016591,1.057672,0.571420,0.092576,-0.162021,-0.165169,0.024407,0.050813,-0.056979,-0.221351,-0.259945,0.038594,0.016769,96.423796


## Step 11 — Split the Data into Training and Testing Sets

I split the prepared dataset into training and testing sets using an 80/20 split. I used stratified sampling to keep the Buy, Sell, and Hold signal proportions similar in both sets. The training data will be used to build the models, while the test data will be used for final evaluation.

In [16]:
# Split Data into Training and Testing Sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data split completed successfully.")

print("\nTraining Set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting Set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining Signal Distribution:")
print(y_train.value_counts())

print("\nTesting Signal Distribution:")
print(y_test.value_counts())

Data split completed successfully.

Training Set:
X_train: (705096, 17)
y_train: (705096,)

Testing Set:
X_test: (176275, 17)
y_test: (176275,)

Training Signal Distribution:
signal
Hold    703589
Buy        905
Sell       602
Name: count, dtype: int64

Testing Signal Distribution:
signal
Hold    175898
Buy        226
Sell       151
Name: count, dtype: int64


## Step 12 — Build the Logistic Regression Model

I trained a Logistic Regression model as the first classification model. Since the trading signals are highly imbalanced, I used balanced class weights so that the smaller Buy and Sell classes receive more attention during training.

In [18]:
# Train Logistic Regression Model

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=3000,
    solver="lbfgs",
    random_state=42
)

logistic_model.fit(X_train, y_train)

# Make predictions
y_pred_logistic = logistic_model.predict(X_test)

print("Logistic Regression model trained successfully.")
print("Predictions completed successfully.")
print("Iterations used:", logistic_model.n_iter_)

Logistic Regression model trained successfully.
Predictions completed successfully.
Iterations used: [1764]


## Step 13 — Evaluate the Logistic Regression Model

I evaluated the Logistic Regression model using accuracy, precision, recall, F1-score, and a confusion matrix. These metrics help show how well the model predicts the Buy, Sell, and Hold trading signals.

In [19]:
# Evaluate Logistic Regression Model

logistic_accuracy = accuracy_score(y_test, y_pred_logistic)

print("Logistic Regression Accuracy:")
print(round(logistic_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic))

print("\nConfusion Matrix:")
logistic_cm = confusion_matrix(
    y_test,
    y_pred_logistic,
    labels=["Buy", "Hold", "Sell"]
)

display(
    pd.DataFrame(
        logistic_cm,
        index=["Actual Buy", "Actual Hold", "Actual Sell"],
        columns=["Predicted Buy", "Predicted Hold", "Predicted Sell"]
    )
)

Logistic Regression Accuracy:
0.7755

Classification Report:
              precision    recall  f1-score   support

         Buy       0.01      0.99      0.03       226
        Hold       1.00      0.78      0.87    175898
        Sell       0.01      0.96      0.01       151

    accuracy                           0.78    176275
   macro avg       0.34      0.91      0.30    176275
weighted avg       1.00      0.78      0.87    176275


Confusion Matrix:


,Predicted Buy,Predicted Hold,Predicted Sell
Actual Buy,223,3,0
Actual Hold,16168,136326,23404
Actual Sell,0,6,145


### Interpretation

The Logistic Regression model achieved 77.55% accuracy. It identified most of the actual Buy and Sell signals, with recall of 99% and 96%. However, precision for these signals was very low because many Hold observations were incorrectly classified as Buy or Sell. This reflects the strong class imbalance in the dataset.

## Step 14 — Build the Random Forest Model

I trained a Random Forest classifier as the second machine learning model. I used balanced class weights to account for the large difference between the number of Hold signals and the smaller Buy and Sell classes.

In [20]:
# Train Random Forest Model

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = random_forest_model.predict(X_test)

print("Random Forest model trained successfully.")
print("Predictions completed successfully.")

Random Forest model trained successfully.
Predictions completed successfully.


## Step 15 — Evaluate the Random Forest Model

I evaluated the Random Forest model using accuracy, precision, recall, F1-score, and a confusion matrix. Using the same evaluation measures makes it easier to compare its performance with the Logistic Regression model.

In [21]:
# Evaluate Random Forest Model

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:")
print(round(rf_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
rf_cm = confusion_matrix(
    y_test,
    y_pred_rf,
    labels=["Buy", "Hold", "Sell"]
)

display(
    pd.DataFrame(
        rf_cm,
        index=["Actual Buy", "Actual Hold", "Actual Sell"],
        columns=["Predicted Buy", "Predicted Hold", "Predicted Sell"]
    )
)

Random Forest Accuracy:
0.9983

Classification Report:
              precision    recall  f1-score   support

         Buy       0.86      0.17      0.28       226
        Hold       1.00      1.00      1.00    175898
        Sell       0.89      0.28      0.42       151

    accuracy                           1.00    176275
   macro avg       0.92      0.48      0.57    176275
weighted avg       1.00      1.00      1.00    176275


Confusion Matrix:


,Predicted Buy,Predicted Hold,Predicted Sell
Actual Buy,38,188,0
Actual Hold,6,175887,5
Actual Sell,0,109,42


### Interpretation

The Random Forest model achieved 99.83% accuracy and performed very well on the Hold class. It also had high precision for Buy and Sell signals, but the recall was low at 17% for Buy and 28% for Sell. This means the model was accurate overall but missed many of the less common trading signals because the dataset is highly imbalanced.

## Step 16 — Build the Support Vector Machine (SVM) Model

I trained a Support Vector Machine as the third classification model. Since the dataset contains a large number of records, I used LinearSVC for better computational efficiency. Balanced class weights were used to help account for the imbalance between Hold, Buy, and Sell signals.

In [24]:
# Train Support Vector Machine Model

svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

svm_model.fit(X_train, y_train)

# Make predictions
y_pred_svm = svm_model.predict(X_test)

print("SVM model trained successfully.")
print("Predictions completed successfully.")

SVM model trained successfully.
Predictions completed successfully.


## Step 17 — Evaluate the Support Vector Machine (SVM) Model

I evaluated the SVM model using accuracy, precision, recall, F1-score, and a confusion matrix. Using the same evaluation metrics allows me to compare the SVM results fairly with Logistic Regression and Random Forest.

In [25]:
# Evaluate SVM Model

svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:")
print(round(svm_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

print("\nConfusion Matrix:")
svm_cm = confusion_matrix(
    y_test,
    y_pred_svm,
    labels=["Buy", "Hold", "Sell"]
)

display(
    pd.DataFrame(
        svm_cm,
        index=["Actual Buy", "Actual Hold", "Actual Sell"],
        columns=["Predicted Buy", "Predicted Hold", "Predicted Sell"]
    )
)

SVM Accuracy:
0.9784

Classification Report:
              precision    recall  f1-score   support

         Buy       0.01      0.11      0.02       226
        Hold       1.00      0.98      0.99    175898
        Sell       0.01      0.08      0.02       151

    accuracy                           0.98    176275
   macro avg       0.34      0.39      0.34    176275
weighted avg       1.00      0.98      0.99    176275


Confusion Matrix:


,Predicted Buy,Predicted Hold,Predicted Sell
Actual Buy,25,201,0
Actual Hold,2182,172436,1280
Actual Sell,0,139,12


### Interpretation

The SVM model achieved 97.84% accuracy and performed very well on the Hold class. However, it had low precision and recall for Buy and Sell signals. This shows that the class imbalance made it difficult for SVM to accurately identify the less common trading signals.

## Step 18 — Model Performance Comparison

I compared Logistic Regression, Random Forest, and SVM using their test-set performance. Since the trading signals are highly imbalanced, accuracy alone is not enough, so the precision, recall, and F1-scores for the individual classes are also important when selecting the best model.

In [26]:
# Compare Model Performance

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "SVM"
    ],
    "Accuracy": [
        logistic_accuracy,
        rf_accuracy,
        svm_accuracy
    ]
})

model_comparison["Accuracy"] = (
    model_comparison["Accuracy"] * 100
).round(2)

model_comparison = model_comparison.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print("Model Accuracy Comparison:")
display(model_comparison)

Model Accuracy Comparison:


,Model,Accuracy
0,Random Forest,99.83
1,SVM,97.84
2,Logistic Regression,77.55


### Interpretation

Random Forest achieved the highest overall accuracy at 99.83%, followed by SVM at 97.84% and Logistic Regression at 77.55%. However, the detailed results show that each model handled the minority Buy and Sell signals differently. Random Forest provided the strongest overall performance, although class imbalance still affected its ability to identify all Buy and Sell signals.

## Step 19 — Random Forest Optimization

Random Forest showed the strongest overall performance, so I selected it for further optimization. I tested several hyperparameter combinations using cross-validation to find settings that provide a better balance between the Buy, Sell, and Hold classes.

In [29]:
# Lightweight Random Forest Optimization

# Create a smaller stratified sample for hyperparameter tuning
X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=50000,
    stratify=y_train,
    random_state=42
)

print("Tuning Sample Shape:", X_tune.shape)
print("\nTuning Signal Distribution:")
print(y_tune.value_counts())

# Small parameter grid
param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [10, 20]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_tune, y_tune)

print("\nRandom Forest optimization completed.")
print("Best Parameters:")
print(rf_grid.best_params_)

print("\nBest Cross-Validation Macro F1:")
print(round(rf_grid.best_score_, 4))

Tuning Sample Shape: (50000, 17)

Tuning Signal Distribution:
signal
Hold    49893
Buy        64
Sell       43
Name: count, dtype: int64
Fitting 3 folds for each of 4 candidates, totalling 12 fits

Random Forest optimization completed.
Best Parameters:
{'max_depth': 10, 'n_estimators': 100}

Best Cross-Validation Macro F1:
0.3819


## Step 20 — Evaluate the Optimized Random Forest

I used the best hyperparameters found during cross-validation to train the Random Forest model on the full training dataset. I then evaluated the optimized model on the test set to see whether the tuning improved its performance.

In [30]:
# Train and Evaluate Optimized Random Forest

optimized_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

optimized_rf.fit(X_train, y_train)

# Predictions
y_pred_optimized_rf = optimized_rf.predict(X_test)

# Accuracy
optimized_rf_accuracy = accuracy_score(
    y_test,
    y_pred_optimized_rf
)

print("Optimized Random Forest Accuracy:")
print(round(optimized_rf_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_optimized_rf))

print("\nConfusion Matrix:")
optimized_rf_cm = confusion_matrix(
    y_test,
    y_pred_optimized_rf,
    labels=["Buy", "Hold", "Sell"]
)

display(
    pd.DataFrame(
        optimized_rf_cm,
        index=["Actual Buy", "Actual Hold", "Actual Sell"],
        columns=["Predicted Buy", "Predicted Hold", "Predicted Sell"]
    )
)

Optimized Random Forest Accuracy:
0.9078

Classification Report:
              precision    recall  f1-score   support

         Buy       0.03      0.93      0.06       226
        Hold       1.00      0.91      0.95    175898
        Sell       0.01      0.91      0.03       151

    accuracy                           0.91    176275
   macro avg       0.35      0.92      0.35    176275
weighted avg       1.00      0.91      0.95    176275


Confusion Matrix:


,Predicted Buy,Predicted Hold,Predicted Sell
Actual Buy,211,15,0
Actual Hold,6713,159666,9519
Actual Sell,0,13,138


### Interpretation

The optimized Random Forest achieved 90.78% accuracy. Although its overall accuracy was lower than the original Random Forest, it identified many more of the actual Buy and Sell signals, with recall of 93% and 91%. This shows that optimization improved the model's ability to detect the minority trading signals, but it also increased false Buy and Sell predictions.

## Step 21 — Final Model Comparison

I compared the performance of all three machine learning models and the optimized Random Forest. This comparison helps show how optimization affected the final model and highlights the trade-off between overall accuracy and detecting the less common Buy and Sell signals.

In [31]:
# Final Model Comparison

final_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "SVM",
        "Optimized Random Forest"
    ],
    "Accuracy (%)": [
        logistic_accuracy * 100,
        rf_accuracy * 100,
        svm_accuracy * 100,
        optimized_rf_accuracy * 100
    ]
})

final_comparison["Accuracy (%)"] = (
    final_comparison["Accuracy (%)"].round(2)
)

print("Final Model Comparison:")
display(final_comparison)

Final Model Comparison:


,Model,Accuracy (%)
0,Logistic Regression,77.55
1,Random Forest,99.83
2,SVM,97.84
3,Optimized Random Forest,90.78


## Step 22 — Final Findings and Financial Implications

The results show that the models performed differently when predicting trading signals. Random Forest achieved the highest overall accuracy at 99.83%, but it missed many Buy and Sell signals. After optimization, the Random Forest detected more of these signals, although its overall accuracy decreased to 90.78%.

From a trading perspective, this shows why accuracy alone should not be used to select a model. Missing a Buy or Sell signal may mean missing a trading opportunity, while a false signal could lead to an unnecessary trade. Therefore, precision, recall, and F1-score should also be considered when using machine learning to support trading decisions.

## Step 23 — Project Conclusion

In this project, I used MACD and RSI indicators to create Buy, Sell, and Hold trading signals. I trained and evaluated Logistic Regression, Random Forest, and SVM models to predict these signals.

The results showed that Random Forest had the highest overall accuracy at 99.83%. However, because the dataset was highly imbalanced, accuracy alone did not fully describe model performance. The optimized Random Forest improved the detection of Buy and Sell signals, showing the importance of considering precision, recall, and F1-score when evaluating trading models.

Overall, this project showed how technical indicators and machine learning can be combined to analyze stock market data and support trading decisions.